# Fluxgym Start - Notebook de Execução Final

In [ ]:
# Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clonar o repositório oficial
!git clone https://github.com/AMPortugal/FluxGymColab.git

In [ ]:
# Instalar as dependências customizadas
!pip install -r FluxGymColab/requirements_custom.txt

In [ ]:
# Reset elegante após instalar as dependências
import os
os.kill(os.getpid(), 9)

In [ ]:
# Reinstalar dependências após reset
!pip install -r FluxGymColab/requirements_custom.txt

In [ ]:
# Identificar GPU e aplicar ajustes de desempenho
import subprocess, os
def detect_gpu():
    gpu_info = subprocess.check_output("nvidia-smi -L", shell=True).decode()
    if "A100" in gpu_info:
        return "A100"
    elif "T4" in gpu_info:
        return "T4"
    else:
        return "Other"
gpu_type = detect_gpu()
print(f"Detected GPU: {gpu_type}")
if gpu_type == "T4":
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
elif gpu_type == "A100":
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

In [ ]:
# Instalar torch/torchaudio/torchvision de acordo com a versão do CUDA
import subprocess, re
cuda_output = subprocess.check_output("nvidia-smi", shell=True).decode("utf-8")
match = re.search(r"CUDA Version: (\d+\.\d+)", cuda_output)
if match:
    cuda_version = match.group(1).replace(".", "")
    index_url = f"https://download.pytorch.org/whl/cu{cuda_version}"
    print(f"Detected CUDA version: {cuda_version}. Installing with index: {index_url}")
    subprocess.run(["pip", "install", "torch", "torchvision", "torchaudio", "--index-url", index_url])
else:
    print("Could not detect CUDA version. Installing CPU-only version.")
    subprocess.run(["pip", "install", "torch", "torchvision", "torchaudio"])

In [ ]:
# Criar ambiente temporário e copiar os núcleos
import shutil, os
TEMP_PATH = '/tmp/fluxgym-flat'
os.makedirs(TEMP_PATH, exist_ok=True)
CORES = ['aitoolkit_core', 'peanut_core', 'sd_core', 'sd3_core']
for core in CORES:
    shutil.copytree(f'/content/FluxGymColab/{core}', f'{TEMP_PATH}/{core}', dirs_exist_ok=True)
shutil.copy('/content/FluxGymColab/requirements_custom.txt', TEMP_PATH)

In [ ]:
# Baixar modelos extras para a pasta temporária
!mkdir -p /tmp/fluxgym-flat/models/unet
!mkdir -p /tmp/fluxgym-flat/models/clip
!mkdir -p /tmp/fluxgym-flat/models/vae
!wget -O /tmp/fluxgym-flat/models/unet/flux1-dev-fp8.safetensors https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors
!wget -O /tmp/fluxgym-flat/models/clip/clip_l.safetensors https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors
!wget -O /tmp/fluxgym-flat/models/clip/t5xxl_fp8.safetensors https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors
!wget -O /tmp/fluxgym-flat/models/vae/ae.sft https://huggingface.co/cocktailpeanut/xulf-dev/resolve/main/ae.sft

In [ ]:
# Gerar app.py e models.yaml dinamicamente
with open('/tmp/fluxgym-flat/models.yaml', 'w') as f:
    f.write('model_list:\n')
    f.write('- name: lora-fruxin\n  path: models/unet/flux1-dev-fp8.safetensors\n')
    f.write('- name: clip-l\n  path: models/clip/clip_l.safetensors\n')
    f.write('- name: t5xxl\n  path: models/clip/t5xxl_fp8.safetensors\n')
    f.write('- name: vae-ae\n  path: models/vae/ae.sft\n')
    
with open('/tmp/fluxgym-flat/app.py', 'w') as f:
    f.write('import gradio as gr\n')
    f.write('def launch():\n')
    f.write('    with open(\'models.yaml\') as f:\n')
    f.write('        models = f.read()\n')
    f.write('    return gr.Interface(fn=lambda: models, inputs=[], outputs=\'textbox\').launch()\n')
    f.write('if __name__ == \"__main__\": launch()\n')

In [ ]:
# Iniciar o app.py via Gradio
!python /tmp/fluxgym-flat/app.py